In [1]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable


 IMPORTS


In [2]:

import os
import re
import json
import pickle
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    ndcg_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

 PATH SETUP


In [3]:

DATA_DIR = Path(r"D:\recommendation_item_API\data")

RAW_FILE = DATA_DIR / "main_data.csv"
ITEM_CATALOG_FILE = DATA_DIR / "item_catalog.csv"
CATEGORY_LOOKUP_FILE = DATA_DIR / "category_embedding_lookup_from_customer_history.csv"

ARTIFACT_DIR = DATA_DIR / "stage1_artifacts_v2"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = DATA_DIR / "model_outputs_v2"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_FILE exists:", RAW_FILE.exists())
print("ITEM_CATALOG_FILE exists:", ITEM_CATALOG_FILE.exists())
print("CATEGORY_LOOKUP_FILE exists:", CATEGORY_LOOKUP_FILE.exists())
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("MODEL_DIR:", MODEL_DIR)

RAW_FILE exists: True
ITEM_CATALOG_FILE exists: True
CATEGORY_LOOKUP_FILE exists: True
ARTIFACT_DIR: C:\D drive\item_recommendation\data\stage1_artifacts_v2
MODEL_DIR: C:\D drive\item_recommendation\data\model_outputs_v2


LOAD FILES


In [4]:

raw_df = pd.read_csv(RAW_FILE)
item_catalog_df = pd.read_csv(ITEM_CATALOG_FILE)
cat_lookup_df = pd.read_csv(CATEGORY_LOOKUP_FILE)

raw_df.columns = [c.strip() for c in raw_df.columns]
item_catalog_df.columns = [c.strip() for c in item_catalog_df.columns]
cat_lookup_df.columns = [c.strip() for c in cat_lookup_df.columns]

print("raw_df shape:", raw_df.shape)
print("item_catalog_df shape:", item_catalog_df.shape)
print("cat_lookup_df shape:", cat_lookup_df.shape)

display(raw_df.head())
display(item_catalog_df.head())
display(cat_lookup_df.head())

raw_df shape: (40886, 32)
item_catalog_df shape: (229, 10)
cat_lookup_df shape: (43, 10)


,Transaction_ID,Customer_ID,Timestamp,Product_Category,Product_Name,Quantity,Price,Total_Amount,Day_Type,transactionId,...,isFestival,isRamadan,isEidPrep,isEid,season,timeSlot,dayType,dayOfWeek,hour,month
0,1,23561,2025-01-01 07:29:00,Dairy-Other,Herman Peanut Butter-340gm,1,154.46,154.46,Weekday,1,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,7,1
1,1,23561,2025-01-01 07:29:00,Bakery-Bread,Pran Toast-250g,4,63.66,254.64,Weekday,1,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,7,1
2,1,23561,2025-01-01 07:29:00,Beverage-Hot,Horlicks Classic Malt-1Kg Jar,1,329.90,329.90,Weekday,1,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,7,1
3,2,23569,2025-01-01 11:33:00,Pantry-DryFood,Lassa Special Semai-500g(Pkt),2,155.51,311.02,Weekday,2,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,11,1
4,2,23569,2025-01-01 11:33:00,Fruits-Fresh,Atta Fruits,2,46.66,93.32,Weekday,2,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,11,1


,itemId,itemName,category,avgPrice,minPrice,maxPrice,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,121.34,121.34,121.34,115,1,1,0
1,27,Pran Toast-250g,Bakery-Bread,63.66,63.66,63.66,244,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Personal-Care-Cosmetics,587.26,587.26,587.26,65,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,122.60,122.60,122.60,43,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,73.72,73.72,73.72,144,1,1,0


,category,cat_emb_0,cat_emb_1,cat_emb_2,cat_emb_3,cat_emb_4,cat_emb_5,cat_emb_6,cat_emb_7,category_embedding
0,Bakery-Bread,30.500512,-0.312345,-1.336538,0.171504,1.265434,-0.221737,-2.422268,-1.024791,"[30.500512, -0.312345, -1.336538, 0.171504, 1...."
1,Beverage-Carbonated,30.793406,0.582054,2.320642,-1.194838,0.826971,-1.544423,1.249234,-0.971547,"[30.793406, 0.582054, 2.320642, -1.194838, 0.8..."
2,Beverage-Hot,29.132634,0.224027,0.067448,-0.667184,-0.053071,0.364866,0.603695,0.191561,"[29.132634, 0.224027, 0.067448, -0.667184, -0...."
3,Beverage-Juice,31.321391,0.378736,0.439785,-0.637943,0.784723,-1.583087,-2.383535,2.169788,"[31.321391, 0.378736, 0.439785, -0.637943, 0.7..."
4,Beverage-Water,22.344446,0.596863,0.331393,-0.024937,0.778449,0.414786,-0.005348,-0.207390,"[22.344446, 0.596863, 0.331393, -0.024937, 0.7..."


 HELPERS


In [5]:

def normalize_text(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x

def normalize_category(cat):
    if pd.isna(cat):
        return cat
    cat = normalize_text(cat)
    parts = [p.strip() for p in cat.split("-")]
    parts = [p.capitalize() if p else p for p in parts]
    return "-".join(parts)

def get_supercategory(category_name):
    if pd.isna(category_name):
        return None
    category_name = str(category_name).strip()
    if "-" in category_name:
        return category_name.split("-")[0]
    return category_name

def week_of_month(dt):
    return ((dt.day - 1) // 7) + 1

def cosine_sim(vec1, vec2):
    if vec1 is None or vec2 is None:
        return 0.0

    v1 = np.array(vec1, dtype=float).reshape(1, -1)
    v2 = np.array(vec2, dtype=float).reshape(1, -1)

    if np.allclose(v1, 0) or np.allclose(v2, 0):
        return 0.0

    dot_val = float(np.dot(v1, v2.T))
    norm_val = float(np.linalg.norm(v1) * np.linalg.norm(v2))
    if norm_val == 0:
        return 0.0
    return dot_val / norm_val

def mean_pool_vectors(vectors):
    valid = [np.array(v, dtype=float) for v in vectors if v is not None]
    if len(valid) == 0:
        return None
    return np.mean(valid, axis=0)

CLEAN RAW DATA


In [6]:

raw_df["itemId"] = pd.to_numeric(raw_df["itemId"], errors="coerce")
raw_df["customerId"] = pd.to_numeric(raw_df["customerId"], errors="coerce")
raw_df["quantity"] = pd.to_numeric(raw_df["quantity"], errors="coerce")
raw_df["price"] = pd.to_numeric(raw_df["price"], errors="coerce")

raw_df["itemName"] = raw_df["itemName"].apply(normalize_text)
raw_df["category"] = raw_df["category"].apply(normalize_category)

raw_df["orderDate"] = pd.to_datetime(raw_df["orderDate"], dayfirst=True, errors="coerce")

raw_df = raw_df.dropna(subset=["itemId", "customerId", "orderDate", "category"]).copy()

raw_df["itemId"] = raw_df["itemId"].astype(int)
raw_df["customerId"] = raw_df["customerId"].astype(int)
raw_df["weekOfMonth"] = raw_df["orderDate"].apply(week_of_month)

print("Cleaned raw shape:", raw_df.shape)
display(raw_df.head())

Cleaned raw shape: (16108, 33)


,Transaction_ID,Customer_ID,Timestamp,Product_Category,Product_Name,Quantity,Price,Total_Amount,Day_Type,transactionId,...,isRamadan,isEidPrep,isEid,season,timeSlot,dayType,dayOfWeek,hour,month,weekOfMonth
0,1,23561,2025-01-01 07:29:00,Dairy-Other,Herman Peanut Butter-340gm,1,154.46,154.46,Weekday,1,...,0,0,0,Winter,Morning,Weekday,Wednesday,7,1,1
1,1,23561,2025-01-01 07:29:00,Bakery-Bread,Pran Toast-250g,4,63.66,254.64,Weekday,1,...,0,0,0,Winter,Morning,Weekday,Wednesday,7,1,1
2,1,23561,2025-01-01 07:29:00,Beverage-Hot,Horlicks Classic Malt-1Kg Jar,1,329.90,329.90,Weekday,1,...,0,0,0,Winter,Morning,Weekday,Wednesday,7,1,1
3,2,23569,2025-01-01 11:33:00,Pantry-DryFood,Lassa Special Semai-500g(Pkt),2,155.51,311.02,Weekday,2,...,0,0,0,Winter,Morning,Weekday,Wednesday,11,1,1
4,2,23569,2025-01-01 11:33:00,Fruits-Fresh,Atta Fruits,2,46.66,93.32,Weekday,2,...,0,0,0,Winter,Morning,Weekday,Wednesday,11,1,1


 BUILD LOOKUPS


In [7]:

item_catalog_df["itemId"] = pd.to_numeric(item_catalog_df["itemId"], errors="coerce")
item_catalog_df = item_catalog_df.dropna(subset=["itemId", "category"]).copy()
item_catalog_df["itemId"] = item_catalog_df["itemId"].astype(int)
item_catalog_df["itemName"] = item_catalog_df["itemName"].apply(normalize_text)
item_catalog_df["category"] = item_catalog_df["category"].apply(normalize_category)

item_to_category = dict(zip(item_catalog_df["itemId"], item_catalog_df["category"]))
item_to_name = dict(zip(item_catalog_df["itemId"], item_catalog_df["itemName"]))

cat_lookup_df["category"] = cat_lookup_df["category"].apply(normalize_category)
cat_lookup_df["category_embedding_vec"] = cat_lookup_df["category_embedding"].apply(json.loads)

category_to_vector = {
    row["category"]: np.array(row["category_embedding_vec"], dtype=float)
    for _, row in cat_lookup_df.iterrows()
}

print("item_to_category size:", len(item_to_category))
print("item_to_name size:", len(item_to_name))
print("category_to_vector size:", len(category_to_vector))

item_to_category size: 229
item_to_name size: 229
category_to_vector size: 43


 BASKET TABLE


In [8]:

basket_df = (
    raw_df
    .sort_values(["transactionId", "orderDate"])
    .groupby("transactionId")
    .agg({
        "customerId": "first",
        "orderDate": "first",
        "season": "first",
        "isHoliday": "max",
        "isFestival": "max",
        "timeSlot": "first",
        "weekOfMonth": "first",
        "itemId": lambda x: list(map(int, x.tolist()))
    })
    .reset_index()
)

basket_df.rename(columns={"itemId": "item_ids"}, inplace=True)

print("Basket shape:", basket_df.shape)
display(basket_df.head())

Basket shape: (2426, 9)


,transactionId,customerId,orderDate,season,isHoliday,isFestival,timeSlot,weekOfMonth,item_ids
0,1,23561,2025-01-01 07:29:00,Winter,0,0,Morning,1,"[7075, 27, 6814]"
1,2,23569,2025-01-01 11:33:00,Winter,0,0,Morning,1,"[25474, 15131, 13786, 7017, 6815, 31849, 13352]"
2,3,23527,2025-01-01 12:39:00,Winter,0,0,Afternoon,1,"[32441, 2306, 13922, 2364, 952, 30743, 32269]"
3,4,23433,2025-01-01 12:46:00,Winter,0,0,Afternoon,1,"[878, 704, 15104, 31328, 14964]"
4,5,23494,2025-01-01 13:21:00,Winter,0,0,Afternoon,1,"[15180, 2220, 2364, 13793, 708, 2043, 12655]"


 INPUT PROFILE AND RULE HELPERS


In [9]:

def get_input_categories(item_ids):
    categories = []
    for iid in item_ids:
        cat = item_to_category.get(int(iid), None)
        if cat is not None:
            categories.append(cat)
    return list(dict.fromkeys(categories))


def get_input_item_names(item_ids):
    names = []
    for iid in item_ids:
        names.append(str(item_to_name.get(int(iid), "")))
    return names


def detect_profile_train(item_ids):
    input_categories = get_input_categories(item_ids)
    input_names = " ".join(get_input_item_names(item_ids)).lower()
    supercats = {get_supercategory(c) for c in input_categories if c is not None}

    cooking_keywords = [
        "beef", "chicken", "potato", "rice", "elachi", "elaichi",
        "cinnamon", "ginger", "fish", "daruchini", "masala",
        "jafran", "saffron"
    ]

    if "Clothing" in supercats or "Footwear" in supercats:
        return "clothing"

    if "Personal" in supercats and "Personal-Care-Sanitary" in input_categories:
        return "sanitary_plus"

    if "Personal" in supercats and "Personal-Care-Cosmetics" in input_categories:
        return "personal_care"

    if any(k in input_names for k in cooking_keywords):
        return "cooking"

    if supercats.intersection({"Meat", "Fish", "Protein", "Veg", "Pantry", "Spices"}):
        return "cooking"

    if supercats.intersection({"Bakery", "Dairy", "Beverage", "Breakfast", "Spreads"}):
        return "breakfast"

    if supercats.intersection({"Snacks", "Chocolates", "Instant"}):
        return "snack"

    return "generic"


def get_profile_rules_train(item_ids):
    profile = detect_profile_train(item_ids)

    if profile == "clothing":
        return {
            "profile_name": profile,
            "allowed_supercategories": {"Clothing", "Footwear"},
            "allowed_categories": {
                "Clothing-Casual",
                "Clothing-Traditional",
                "Clothing-Accessories",
                "Clothing-Innerwear",
                "Footwear-Casual",
                "Footwear-Shoes"
            },
            "preferred_categories": {
                "Clothing-Casual",
                "Clothing-Traditional",
                "Clothing-Accessories",
                "Footwear-Casual",
                "Footwear-Shoes"
            },
            "required_any_groups": [
                {"Clothing-Casual", "Clothing-Traditional"},
                {"Clothing-Accessories", "Footwear-Casual", "Footwear-Shoes"}
            ]
        }

    if profile == "personal_care":
        return {
            "profile_name": profile,
            "allowed_supercategories": {"Personal"},
            "allowed_categories": {
                "Personal-Care-Bath",
                "Personal-Care-Hair",
                "Personal-Care-Oral",
                "Personal-Care-Cosmetics",
                "Personal-Care-Sanitary"
            },
            "preferred_categories": {
                "Personal-Care-Bath",
                "Personal-Care-Hair",
                "Personal-Care-Oral",
                "Personal-Care-Cosmetics"
            },
            "required_any_groups": [
                {"Personal-Care-Bath", "Personal-Care-Hair", "Personal-Care-Oral", "Personal-Care-Cosmetics"}
            ]
        }

    if profile == "sanitary_plus":
        return {
            "profile_name": profile,
            "allowed_supercategories": {"Personal", "Fruits", "Beverage", "Dairy"},
            "allowed_categories": {
                "Personal-Care-Sanitary",
                "Personal-Care-Bath",
                "Personal-Care-Hair",
                "Personal-Care-Oral",
                "Personal-Care-Cosmetics",
                "Fruits-Fresh",
                "Beverage-Hot",
                "Dairy-Milk",
                "Dairy-Other"
            },
            "preferred_categories": {
                "Personal-Care-Sanitary",
                "Fruits-Fresh",
                "Beverage-Hot",
                "Dairy-Other",
                "Dairy-Milk"
            },
            "required_any_groups": [
                {"Fruits-Fresh"},
                {"Beverage-Hot", "Dairy-Milk", "Dairy-Other"}
            ]
        }

    if profile == "cooking":
        return {
            "profile_name": profile,
            "allowed_supercategories": {"Meat", "Fish", "Protein", "Veg", "Pantry", "Spices", "Beverage", "Fruits", "Dairy"},
            "allowed_categories": {
                "Spices-Cooking",
                "Pantry-Oils",
                "Veg-Cooking",
                "Pantry-Rice",
                "Pantry-Pulses",
                "Pantry-DryFood",
                "Protein-Egg",
                "Beverage-Carbonated",
                "Beverage-Juice",
                "Beverage-Hot",
                "Meat-Fresh",
                "Fish-Fresh",
                "Dairy-Other",
                "Fruits-Fresh"
            },
            "preferred_categories": {
                "Pantry-Oils",
                "Veg-Cooking",
                "Pantry-Rice",
                "Spices-Cooking",
                "Protein-Egg",
                "Beverage-Carbonated",
                "Beverage-Juice",
                "Beverage-Hot"
            },
            "required_any_groups": [
                {"Protein-Egg"},
                {"Beverage-Juice", "Beverage-Carbonated", "Beverage-Hot"}
            ]
        }

    if profile == "breakfast":
        return {
            "profile_name": profile,
            "allowed_supercategories": {"Bakery", "Dairy", "Beverage", "Breakfast", "Spreads", "Protein"},
            "allowed_categories": {
                "Bakery-Bread",
                "Beverage-Hot",
                "Dairy-Milk",
                "Dairy-Other",
                "Spreads",
                "Breakfast-Cereal",
                "Protein-Egg"
            },
            "preferred_categories": {
                "Bakery-Bread",
                "Beverage-Hot",
                "Dairy-Milk",
                "Spreads",
                "Protein-Egg"
            },
            "required_any_groups": [
                {"Beverage-Hot"},
                {"Bakery-Bread", "Spreads"}
            ]
        }

    if profile == "snack":
        return {
            "profile_name": profile,
            "allowed_supercategories": {"Snacks", "Beverage", "Instant", "Chocolates", "Bakery"},
            "allowed_categories": {
                "Snacks-General",
                "Beverage-Carbonated",
                "Beverage-Juice",
                "Instant-Food",
                "Chocolates-Sweets",
                "Bakery-Bread"
            },
            "preferred_categories": {
                "Snacks-General",
                "Beverage-Carbonated",
                "Beverage-Juice",
                "Instant-Food",
                "Chocolates-Sweets"
            },
            "required_any_groups": [
                {"Beverage-Carbonated", "Beverage-Juice"},
                {"Snacks-General", "Instant-Food"}
            ]
        }

    return {
        "profile_name": "generic",
        "allowed_supercategories": set(),
        "allowed_categories": set(),
        "preferred_categories": set(),
        "required_any_groups": []
    }


def candidate_allowed_by_profile(candidate_category, profile_rules):
    if candidate_category is None:
        return 0

    candidate_supercat = get_supercategory(candidate_category)

    allowed_exact = profile_rules["allowed_categories"]
    allowed_supercats = profile_rules["allowed_supercategories"]

    if len(allowed_exact) == 0 and len(allowed_supercats) == 0:
        return 1

    if candidate_category in allowed_exact:
        return 1

    if candidate_supercat in allowed_supercats:
        return 1

    return 0


def candidate_preferred_by_profile(candidate_category, profile_rules):
    return 1 if candidate_category in profile_rules["preferred_categories"] else 0


def candidate_required_group_flag(candidate_category, profile_rules):
    for grp in profile_rules["required_any_groups"]:
        if candidate_category in grp:
            return 1
    return 0

 STAGE 1 RULES


In [10]:

basket_items_for_rules = basket_df["item_ids"].apply(
    lambda x: [str(v) for v in sorted(set(x))]
).tolist()

te = TransactionEncoder()
basket_matrix = te.fit(basket_items_for_rules).transform(basket_items_for_rules)
basket_matrix_df = pd.DataFrame(basket_matrix, columns=te.columns_)

freq_items = fpgrowth(
    basket_matrix_df,
    min_support=0.005,
    use_colnames=True,
    max_len=3
)

freq_items["itemset_len"] = freq_items["itemsets"].apply(len)

rules_df = association_rules(
    freq_items,
    metric="confidence",
    min_threshold=0.10
)

rules_df["antecedents"] = rules_df["antecedents"].apply(lambda s: sorted(list(s)))
rules_df["consequents"] = rules_df["consequents"].apply(lambda s: sorted(list(s)))

print("Frequent itemsets shape:", freq_items.shape)
print("Rules shape:", rules_df.shape)
display(rules_df.head())

Frequent itemsets shape: (704, 3)
Rules shape: (713, 14)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,[3831],[6814],0.044930,0.040396,0.006183,0.137615,3.406665,1.0,0.004368,1.112733,0.739692,0.078125,0.101312,0.145338
1,[6814],[3831],0.040396,0.044930,0.006183,0.153061,3.406665,1.0,0.004368,1.127673,0.736197,0.078125,0.113218,0.145338
2,[6814],[13352],0.040396,0.087799,0.007007,0.173469,1.975759,1.0,0.003461,1.103651,0.514655,0.057823,0.093916,0.126641
3,[6814],[14034],0.040396,0.098516,0.005359,0.132653,1.346512,1.0,0.001379,1.039358,0.268173,0.040123,0.037868,0.093523
4,[27],[6814],0.037510,0.040396,0.007832,0.208791,5.168648,1.0,0.006317,1.212833,0.837958,0.111765,0.175484,0.201334


CO PURCHASE AND HISTORY COUNTERS


In [11]:

item_pair_counter = Counter()
context_item_counter = Counter()
user_item_counter = Counter()
user_category_counter = Counter()
category_item_counter = Counter()

for _, row in basket_df.iterrows():
    items = sorted(set(row["item_ids"]))
    n = len(items)

    for i in range(n):
        for j in range(i + 1, n):
            a = items[i]
            b = items[j]
            item_pair_counter[(a, b)] += 1
            item_pair_counter[(b, a)] += 1

for _, row in raw_df.iterrows():
    context_key = (
        row["season"],
        int(row["isHoliday"]),
        int(row["isFestival"]),
        row["timeSlot"],
        int(row["weekOfMonth"])
    )
    context_item_counter[(context_key, int(row["itemId"]))] += 1

    user_id = int(row["customerId"])
    item_id = int(row["itemId"])
    category = row["category"]

    user_item_counter[(user_id, item_id)] += 1
    user_category_counter[(user_id, category)] += 1
    category_item_counter[(category, item_id)] += 1

print("Directional item pairs:", len(item_pair_counter))
print("Context counter size:", len(context_item_counter))
print("User item counter size:", len(user_item_counter))
print("User category counter size:", len(user_category_counter))
print("Category item counter size:", len(category_item_counter))

Directional item pairs: 30024
Context counter size: 6591
User item counter size: 7275
User category counter size: 3115
Category item counter size: 229


 SAVE STAGE 1 ARTIFACTS


In [12]:

with open(ARTIFACT_DIR / "association_rules.pkl", "wb") as f:
    pickle.dump(rules_df, f)

with open(ARTIFACT_DIR / "item_pair_counter.pkl", "wb") as f:
    pickle.dump(item_pair_counter, f)

with open(ARTIFACT_DIR / "context_item_counter.pkl", "wb") as f:
    pickle.dump(context_item_counter, f)

with open(ARTIFACT_DIR / "user_item_counter.pkl", "wb") as f:
    pickle.dump(user_item_counter, f)

with open(ARTIFACT_DIR / "user_category_counter.pkl", "wb") as f:
    pickle.dump(user_category_counter, f)

with open(ARTIFACT_DIR / "category_item_counter.pkl", "wb") as f:
    pickle.dump(category_item_counter, f)

with open(ARTIFACT_DIR / "item_to_category.pkl", "wb") as f:
    pickle.dump(item_to_category, f)

with open(ARTIFACT_DIR / "item_to_name.pkl", "wb") as f:
    pickle.dump(item_to_name, f)

with open(ARTIFACT_DIR / "category_to_vector.pkl", "wb") as f:
    pickle.dump(category_to_vector, f)

print("Stage 1 artifacts saved.")

Stage 1 artifacts saved.


 FEATURE BUILD HELPERS


In [13]:

def build_context_embedding_from_items(item_ids):
    cats = []
    for iid in item_ids:
        cat = item_to_category.get(int(iid), None)
        if cat is not None:
            cats.append(cat)

    cats = list(dict.fromkeys(cats))
    vecs = [category_to_vector.get(cat, None) for cat in cats]
    return mean_pool_vectors(vecs)


def get_rule_candidates(context_item_ids, top_n=20):
    context_set = set([str(int(x)) for x in context_item_ids])
    score_counter = Counter()

    for _, row in rules_df.iterrows():
        antecedents = set(row["antecedents"])
        consequents = [int(x) for x in row["consequents"]]

        if antecedents.issubset(context_set):
            score = float(row["confidence"]) * float(row["lift"])
            for item in consequents:
                score_counter[item] += score

    return [item for item, _ in score_counter.most_common(top_n)]


def get_top_copurchase_candidates(context_item_ids, top_n=20):
    score_counter = Counter()

    for ctx_item in context_item_ids:
        for (a, b), cnt in item_pair_counter.items():
            if b == int(ctx_item):
                score_counter[a] += cnt

    return [item for item, _ in score_counter.most_common(top_n)]


def get_top_context_candidates(context_key, top_n=20):
    score_counter = Counter()

    for (ctx_key, item_id), cnt in context_item_counter.items():
        if ctx_key == context_key:
            score_counter[item_id] += cnt

    return [item for item, _ in score_counter.most_common(top_n)]


def get_top_user_history_candidates(customer_id, top_n=20):
    score_counter = Counter()

    for (uid, item_id), cnt in user_item_counter.items():
        if uid == customer_id:
            score_counter[item_id] += cnt

    return [item for item, _ in score_counter.most_common(top_n)]


def get_top_items_from_category(category_name, exclude_items=None, top_n=3):
    if exclude_items is None:
        exclude_items = set()

    score_counter = Counter()

    for (cat, item_id), cnt in category_item_counter.items():
        if cat == category_name and item_id not in exclude_items:
            score_counter[item_id] += cnt

    return [item for item, _ in score_counter.most_common(top_n)]


def build_candidate_pool(customer_id, context_item_ids, context_key, top_n_each=20):
    rule_candidates = get_rule_candidates(context_item_ids, top_n=top_n_each)
    cp_candidates = get_top_copurchase_candidates(context_item_ids, top_n=top_n_each)
    ctx_candidates = get_top_context_candidates(context_key, top_n=top_n_each)
    hist_candidates = get_top_user_history_candidates(customer_id, top_n=top_n_each)

    candidate_counter = Counter()

    for rank, item in enumerate(rule_candidates):
        candidate_counter[item] += 1.5 / (rank + 1)

    for rank, item in enumerate(cp_candidates):
        candidate_counter[item] += 1.3 / (rank + 1)

    for rank, item in enumerate(ctx_candidates):
        candidate_counter[item] += 0.8 / (rank + 1)

    for rank, item in enumerate(hist_candidates):
        candidate_counter[item] += 0.7 / (rank + 1)

    profile_rules = get_profile_rules_train(context_item_ids)

    filtered_candidates = []
    filtered_set = set()

    for item, _ in candidate_counter.most_common(150):
        if item in context_item_ids:
            continue

        candidate_category = item_to_category.get(int(item), None)
        if candidate_allowed_by_profile(candidate_category, profile_rules) == 1:
            if item not in filtered_set:
                filtered_candidates.append(item)
                filtered_set.add(item)

    # required groups inject
    existing_categories = {item_to_category.get(int(i), None) for i in filtered_candidates}

    for grp in profile_rules["required_any_groups"]:
        group_hit = False
        for cat in existing_categories:
            if cat in grp:
                group_hit = True
                break

        if not group_hit:
            for cat in grp:
                extra_items = get_top_items_from_category(
                    category_name=cat,
                    exclude_items=set(context_item_ids).union(filtered_set),
                    top_n=3
                )
                if len(extra_items) > 0:
                    chosen = extra_items[0]
                    filtered_candidates.append(chosen)
                    filtered_set.add(chosen)
                    break

    return filtered_candidates[:100]

 RANKER FEATURE HELPERS


In [14]:

def get_item_cooccurrence_score(candidate_item_id, context_item_ids):
    if len(context_item_ids) == 0:
        return 0.0
    scores = [item_pair_counter.get((int(candidate_item_id), int(ctx)), 0) for ctx in context_item_ids]
    return float(np.mean(scores)) if len(scores) > 0 else 0.0

def get_customer_history_score(customer_id, candidate_item_id):
    return float(user_item_counter.get((int(customer_id), int(candidate_item_id)), 0))

def get_category_affinity_score(customer_id, candidate_item_id):
    cat = item_to_category.get(int(candidate_item_id), None)
    if cat is None:
        return 0.0
    return float(user_category_counter.get((int(customer_id), cat), 0))

def get_context_popularity_score(context_key, candidate_item_id):
    return float(context_item_counter.get((context_key, int(candidate_item_id)), 0))

def get_embedding_similarity_score(context_embedding_vec, candidate_item_id):
    cat = item_to_category.get(int(candidate_item_id), None)
    cand_vec = category_to_vector.get(cat, None) if cat is not None else None
    return cosine_sim(context_embedding_vec, cand_vec)

BUILD ONE TRAINING ROW


In [15]:

def build_ranker_row(
    customer_id,
    context_item_ids,
    candidate_item_id,
    label,
    season,
    is_holiday,
    is_festival,
    time_slot,
    week_of_month
):
    context_key = (
        season,
        int(is_holiday),
        int(is_festival),
        time_slot,
        int(week_of_month)
    )

    context_embedding_vec = build_context_embedding_from_items(context_item_ids)

    input_categories = get_input_categories(context_item_ids)
    profile_rules = get_profile_rules_train(context_item_ids)
    profile_name = profile_rules["profile_name"]

    candidate_category = item_to_category.get(int(candidate_item_id), "Unknown")

    allowed_flag = candidate_allowed_by_profile(candidate_category, profile_rules)
    preferred_flag = candidate_preferred_by_profile(candidate_category, profile_rules)
    required_group_flag = candidate_required_group_flag(candidate_category, profile_rules)

    business_boost = (
        0.10 * allowed_flag +
        0.20 * preferred_flag +
        0.20 * required_group_flag
    )

    return {
        "customerId": int(customer_id),
        "candidate_item_id": int(candidate_item_id),
        "candidate_category": candidate_category,
        "basket_size": len(context_item_ids),
        "item_cooccurrence_score": get_item_cooccurrence_score(candidate_item_id, context_item_ids),
        "category_affinity_score": get_category_affinity_score(customer_id, candidate_item_id),
        "context_popularity_score": get_context_popularity_score(context_key, candidate_item_id),
        "customer_history_score": get_customer_history_score(customer_id, candidate_item_id),
        "embedding_similarity_score": get_embedding_similarity_score(context_embedding_vec, candidate_item_id),
        "business_boost_score": business_boost,
        "allowed_category_flag": allowed_flag,
        "preferred_category_flag": preferred_flag,
        "required_group_flag": required_group_flag,
        "input_category_count": len(input_categories),
        "profile_name": profile_name,
        "season": season,
        "isHoliday": int(is_holiday),
        "isFestival": int(is_festival),
        "timeSlot": time_slot,
        "weekOfMonth": int(week_of_month),
        "label": int(label)
    }

BUILD IMPROVED RANKER TRAINING DATASET


In [ ]:

training_rows = []
negative_per_positive = 10
group_counter = 0

for _, row in basket_df.iterrows():
    basket_items = list(dict.fromkeys(row["item_ids"]))

    if len(basket_items) < 2:
        continue

    customer_id = row["customerId"]
    season = row["season"]
    is_holiday = row["isHoliday"]
    is_festival = row["isFestival"]
    time_slot = row["timeSlot"]
    week_of_month = row["weekOfMonth"]

    for target_item in basket_items:
        context_item_ids = [x for x in basket_items if x != target_item]

        context_key = (
            season,
            int(is_holiday),
            int(is_festival),
            time_slot,
            int(week_of_month)
        )

        candidate_pool = build_candidate_pool(
            customer_id=customer_id,
            context_item_ids=context_item_ids,
            context_key=context_key,
            top_n_each=20
        )

        if target_item not in candidate_pool:
            candidate_pool.append(target_item)

        candidate_pool = [x for x in candidate_pool if x not in context_item_ids]

        group_counter += 1
        group_id = group_counter

        pos_row = build_ranker_row(
            customer_id=customer_id,
            context_item_ids=context_item_ids,
            candidate_item_id=target_item,
            label=1,
            season=season,
            is_holiday=is_holiday,
            is_festival=is_festival,
            time_slot=time_slot,
            week_of_month=week_of_month
        )
        pos_row["group_id"] = group_id
        training_rows.append(pos_row)

        negative_candidates = [x for x in candidate_pool if x != target_item]

        if len(negative_candidates) > negative_per_positive:
            negative_candidates = random.sample(negative_candidates, negative_per_positive)

        for neg_item in negative_candidates:
            neg_row = build_ranker_row(
                customer_id=customer_id,
                context_item_ids=context_item_ids,
                candidate_item_id=neg_item,
                label=0,
                season=season,
                is_holiday=is_holiday,
                is_festival=is_festival,
                time_slot=time_slot,
                week_of_month=week_of_month
            )
            neg_row["group_id"] = group_id
            training_rows.append(neg_row)

ranker_df = pd.DataFrame(training_rows)

print("Improved ranker dataset shape:", ranker_df.shape)
display(ranker_df.head(20))

C:\Users\HP\AppData\Local\Temp\ipykernel_13148\342000514.py:40: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  dot_val = float(np.dot(v1, v2.T))
C:\Users\HP\AppData\Local\Temp\ipykernel_13148\342000514.py:40: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  dot_val = float(np.dot(v1, v2.T))
C:\Users\HP\AppData\Local\Temp\ipykernel_13148\342000514.py:40: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  dot_val = float(np.dot(v1, v2.T))
C:\Users\HP\AppData\Local\Temp\ipyker

 SAVE RANKER DATASET


In [17]:

RANKER_FILE = DATA_DIR / "ranker_training_dataset_v2.csv"
ranker_df.to_csv(RANKER_FILE, index=False)

print("Saved:", RANKER_FILE)

Saved: C:\D drive\item_recommendation\data\ranker_training_dataset_v2.csv


 TRAIN VALID TEST SPLIT BY GROUP


In [18]:

all_groups = ranker_df["group_id"].drop_duplicates().tolist()
random.seed(42)
random.shuffle(all_groups)

n_groups = len(all_groups)
train_end = int(n_groups * 0.80)
valid_end = int(n_groups * 0.90)

train_groups = set(all_groups[:train_end])
valid_groups = set(all_groups[train_end:valid_end])
test_groups = set(all_groups[valid_end:])

train_df = ranker_df[ranker_df["group_id"].isin(train_groups)].copy()
valid_df = ranker_df[ranker_df["group_id"].isin(valid_groups)].copy()
test_df = ranker_df[ranker_df["group_id"].isin(test_groups)].copy()

print("Train shape:", train_df.shape, "groups:", train_df["group_id"].nunique())
print("Valid shape:", valid_df.shape, "groups:", valid_df["group_id"].nunique())
print("Test shape :", test_df.shape, "groups:", test_df["group_id"].nunique())

Train shape: (141724, 20) groups: 12884
Valid shape: (17710, 20) groups: 1610
Test shape : (17721, 20) groups: 1611


FEATURE COLUMNS


In [19]:

feature_cols = [
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "business_boost_score",
    "allowed_category_flag",
    "preferred_category_flag",
    "required_group_flag",
    "input_category_count",
    "isHoliday",
    "isFestival",
    "weekOfMonth",
    "season",
    "timeSlot",
    "candidate_category",
    "profile_name"
]

target_col = "label"
group_col = "group_id"

print(feature_cols)

['basket_size', 'item_cooccurrence_score', 'category_affinity_score', 'context_popularity_score', 'customer_history_score', 'embedding_similarity_score', 'business_boost_score', 'allowed_category_flag', 'input_category_count', 'isHoliday', 'isFestival', 'weekOfMonth', 'season', 'timeSlot', 'candidate_category', 'meal_intent']


ONE HOT ENCODE


In [20]:

X_train_raw = train_df[feature_cols].copy()
X_valid_raw = valid_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()

X_train = pd.get_dummies(
    X_train_raw,
    columns=["season", "timeSlot", "candidate_category", "profile_name"]
)
X_valid = pd.get_dummies(
    X_valid_raw,
    columns=["season", "timeSlot", "candidate_category", "profile_name"]
)
X_test = pd.get_dummies(
    X_test_raw,
    columns=["season", "timeSlot", "candidate_category", "profile_name"]
)

X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df[target_col].astype(int).values
y_valid = valid_df[target_col].astype(int).values
y_test = test_df[target_col].astype(int).values

qid_train = train_df[group_col].values
qid_valid = valid_df[group_col].values
qid_test = test_df[group_col].values

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test shape :", X_test.shape)

X_train shape: (141724, 66)
X_valid shape: (17710, 66)
X_test shape : (17721, 66)


 SORT BY QID


In [21]:

def sort_by_qid(X, y, qid):
    temp = X.copy()
    temp["_label"] = y
    temp["_qid"] = qid

    temp = temp.sort_values("_qid").reset_index(drop=True)

    y_sorted = temp["_label"].values
    qid_sorted = temp["_qid"].values
    X_sorted = temp.drop(columns=["_label", "_qid"])

    return X_sorted, y_sorted, qid_sorted

X_train_sorted, y_train_sorted, qid_train_sorted = sort_by_qid(X_train, y_train, qid_train)
X_valid_sorted, y_valid_sorted, qid_valid_sorted = sort_by_qid(X_valid, y_valid, qid_valid)
X_test_sorted, y_test_sorted, qid_test_sorted = sort_by_qid(X_test, y_test, qid_test)

print(X_train_sorted.shape, X_valid_sorted.shape, X_test_sorted.shape)

(141724, 66) (17710, 66) (17721, 66)


TRAIN UPDATED XGBOOST RANKER


In [22]:

ranker = xgb.XGBRanker(
    objective="rank:ndcg",
    eval_metric="ndcg@10",
    tree_method="hist",
    learning_rate=0.05,
    max_depth=6,
    n_estimators=350,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    lambdarank_pair_method="topk",
    lambdarank_num_pair_per_sample=8
)

ranker.fit(
    X_train_sorted,
    y_train_sorted,
    qid=qid_train_sorted,
    eval_set=[(X_valid_sorted, y_valid_sorted)],
    eval_qid=[qid_valid_sorted],
    verbose=True
)

[0]	validation_0-ndcg@10:0.40944
[1]	validation_0-ndcg@10:0.55715
[2]	validation_0-ndcg@10:0.60099
[3]	validation_0-ndcg@10:0.62152
[4]	validation_0-ndcg@10:0.62804
[5]	validation_0-ndcg@10:0.63083
[6]	validation_0-ndcg@10:0.63036
[7]	validation_0-ndcg@10:0.64070
[8]	validation_0-ndcg@10:0.65821
[9]	validation_0-ndcg@10:0.66132
[10]	validation_0-ndcg@10:0.68357
[11]	validation_0-ndcg@10:0.68362
[12]	validation_0-ndcg@10:0.68485
[13]	validation_0-ndcg@10:0.68755
[14]	validation_0-ndcg@10:0.68781
[15]	validation_0-ndcg@10:0.69013
[16]	validation_0-ndcg@10:0.69145
[17]	validation_0-ndcg@10:0.69180
[18]	validation_0-ndcg@10:0.69519
[19]	validation_0-ndcg@10:0.69549
[20]	validation_0-ndcg@10:0.69681
[21]	validation_0-ndcg@10:0.70380
[22]	validation_0-ndcg@10:0.70522
[23]	validation_0-ndcg@10:0.70607
[24]	validation_0-ndcg@10:0.70571
[25]	validation_0-ndcg@10:0.70426
[26]	validation_0-ndcg@10:0.70918
[27]	validation_0-ndcg@10:0.70850
[28]	validation_0-ndcg@10:0.70843
[29]	validation_0-ndcg@1

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'rank:ndcg'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sk

 PREDICT


In [23]:

train_meta = train_df.copy().sort_values(group_col).reset_index(drop=True)
valid_meta = valid_df.copy().sort_values(group_col).reset_index(drop=True)
test_meta = test_df.copy().sort_values(group_col).reset_index(drop=True)

train_meta["pred_score"] = ranker.predict(X_train_sorted)
valid_meta["pred_score"] = ranker.predict(X_valid_sorted)
test_meta["pred_score"] = ranker.predict(X_test_sorted)

display(test_meta.head())

,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,business_boost_score,...,meal_intent,input_category_count,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id,pred_score
0,23569,7017,Dairy-Other,6,6.000000,20.0,2.0,4.0,0.992456,0.00,...,cooking,6,Winter,0,0,Morning,1,1,7,1.492725
1,23569,3191,Beverage-Juice,6,6.166667,7.0,1.0,4.0,0.991515,0.06,...,cooking,6,Winter,0,0,Morning,1,0,7,-0.125951
2,23569,13793,Meat-Fresh,6,9.500000,19.0,2.0,0.0,0.983442,0.00,...,cooking,6,Winter,0,0,Morning,1,0,7,-5.033515
3,23569,7075,Dairy-Other,6,3.000000,20.0,5.0,2.0,0.992456,0.00,...,cooking,6,Winter,0,0,Morning,1,0,7,0.189515
4,23569,28573,Household-Aircare,6,4.333333,1.0,5.0,1.0,0.998735,0.00,...,cooking,6,Winter,0,0,Morning,1,0,7,-0.435207


 METRIC HELPERS


In [24]:

def precision_at_k(y_true, y_score, k=5):
    order = np.argsort(y_score)[::-1][:k]
    topk_true = np.array(y_true)[order]
    return float(np.sum(topk_true)) / float(k)

def recall_at_k(y_true, y_score, k=5):
    y_true = np.array(y_true)
    total_relevant = np.sum(y_true)
    if total_relevant == 0:
        return 0.0
    order = np.argsort(y_score)[::-1][:k]
    topk_true = y_true[order]
    return float(np.sum(topk_true)) / float(total_relevant)

def ndcg_at_k(y_true, y_score, k=5):
    y_true = np.array(y_true).reshape(1, -1)
    y_score = np.array(y_score).reshape(1, -1)
    return float(ndcg_score(y_true, y_score, k=k))

def evaluate_grouped_ranking(df_scored, group_col="group_id", label_col="label", score_col="pred_score", k=5):
    precision_scores = []
    recall_scores = []
    ndcg_scores = []

    for gid, grp in df_scored.groupby(group_col):
        y_true = grp[label_col].values
        y_score = grp[score_col].values

        precision_scores.append(precision_at_k(y_true, y_score, k=k))
        recall_scores.append(recall_at_k(y_true, y_score, k=k))
        ndcg_scores.append(ndcg_at_k(y_true, y_score, k=k))

    return {
        f"Precision@{k}": float(np.mean(precision_scores)),
        f"Recall@{k}": float(np.mean(recall_scores)),
        f"NDCG@{k}": float(np.mean(ndcg_scores))
    }

def build_top1_binary_view(df_scored, group_col="group_id", label_col="label", score_col="pred_score"):
    rows = []
    for gid, grp in df_scored.groupby(group_col):
        grp = grp.copy().sort_values(score_col, ascending=False).reset_index(drop=True)
        grp["pred_label_top1"] = 0
        if len(grp) > 0:
            grp.loc[0, "pred_label_top1"] = 1
        rows.append(grp)
    return pd.concat(rows, axis=0).reset_index(drop=True)

def classification_metrics(df_view):
    y_true = df_view["label"].astype(int).values
    y_pred = df_view["pred_label_top1"].astype(int).values
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_true, y_pred, zero_division=0))
    }

 SHOW METRICS


In [25]:

train_cls = build_top1_binary_view(train_meta)
valid_cls = build_top1_binary_view(valid_meta)
test_cls = build_top1_binary_view(test_meta)

for k in [3, 5, 10]:
    print(f"\n===== Ranking Metrics at K = {k} =====")
    print("Train:", evaluate_grouped_ranking(train_meta, k=k))
    print("Valid:", evaluate_grouped_ranking(valid_meta, k=k))
    print("Test :", evaluate_grouped_ranking(test_meta, k=k))

print("\n===== Auxiliary Classification Metrics =====")
print("Train:", classification_metrics(train_cls))
print("Valid:", classification_metrics(valid_cls))
print("Test :", classification_metrics(test_cls))


===== Ranking Metrics at K = 3 =====
Train: {'Precision@3': 0.28717789506364483, 'Recall@3': 0.8615336851909345, 'NDCG@3': 0.7635750739681322}
Valid: {'Precision@3': 0.27184265010351966, 'Recall@3': 0.815527950310559, 'NDCG@3': 0.7100377678105637}
Test : {'Precision@3': 0.2747775708669563, 'Recall@3': 0.824332712600869, 'NDCG@3': 0.7081040858051836}

===== Ranking Metrics at K = 5 =====
Train: {'Precision@5': 0.19023595156783607, 'Recall@5': 0.9511797578391804, 'NDCG@5': 0.8006052823029306}
Valid: {'Precision@5': 0.18509316770186335, 'Recall@5': 0.9254658385093167, 'NDCG@5': 0.7555865893934636}
Test : {'Precision@5': 0.18708876474239602, 'Recall@5': 0.9354438237119801, 'NDCG@5': 0.7543225205473616}

===== Ranking Metrics at K = 10 =====
Train: {'Precision@10': 0.0999689537410742, 'Recall@10': 0.999689537410742, 'NDCG@10': 0.8169883713415078}
Valid: {'Precision@10': 0.09993788819875776, 'Recall@10': 0.9993788819875776, 'NDCG@10': 0.780550532704836}
Test : {'Precision@10': 0.09987585350

# =========================================
# 25. SAVE MODEL AND FEATURES


In [26]:

MODEL_FILE = MODEL_DIR / "xgboost_ranker_model.json"
FEATURE_FILE = MODEL_DIR / "ranker_feature_columns.json"
IMPORTANCE_FILE = MODEL_DIR / "ranker_feature_importance.csv"
SUMMARY_FILE = MODEL_DIR / "training_summary.json"

ranker.save_model(MODEL_FILE)

with open(FEATURE_FILE, "w", encoding="utf-8") as f:
    json.dump(list(X_train_sorted.columns), f, ensure_ascii=False, indent=2)

feature_importance_df = pd.DataFrame({
    "feature": X_train_sorted.columns,
    "importance": ranker.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance_df.to_csv(IMPORTANCE_FILE, index=False)

summary = {
    "valid_ranking_at_10": evaluate_grouped_ranking(valid_meta, k=10),
    "test_ranking_at_10": evaluate_grouped_ranking(test_meta, k=10),
    "valid_aux_metrics": classification_metrics(valid_cls),
    "test_aux_metrics": classification_metrics(test_cls),
    "eval_history": ranker.evals_result()
}

with open(SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved model:", MODEL_FILE)
print("Saved feature file:", FEATURE_FILE)
print("Saved importance file:", IMPORTANCE_FILE)
print("Saved summary:", SUMMARY_FILE)

Saved model: C:\D drive\item_recommendation\data\model_outputs_v2\xgboost_ranker_model.json
Saved feature file: C:\D drive\item_recommendation\data\model_outputs_v2\ranker_feature_columns.json
Saved importance file: C:\D drive\item_recommendation\data\model_outputs_v2\ranker_feature_importance.csv
Saved summary: C:\D drive\item_recommendation\data\model_outputs_v2\training_summary.json


In [27]:
# =========================================
# 26. SAVE FINAL RESULT CSV
# =========================================
summary_rows = []

for split_name, df_scored, cls_df in [
    ("Train", train_meta, train_cls),
    ("Valid", valid_meta, valid_cls),
    ("Test", test_meta, test_cls)
]:
    rank5 = evaluate_grouped_ranking(df_scored, k=5)
    rank10 = evaluate_grouped_ranking(df_scored, k=10)
    cls = classification_metrics(cls_df)

    summary_rows.append({
        "Split": split_name,
        "Precision@5": rank5["Precision@5"],
        "Recall@5": rank5["Recall@5"],
        "NDCG@5": rank5["NDCG@5"],
        "Precision@10": rank10["Precision@10"],
        "Recall@10": rank10["Recall@10"],
        "NDCG@10": rank10["NDCG@10"],
        "Accuracy": cls["accuracy"],
        "Precision_cls": cls["precision"],
        "Recall_cls": cls["recall"],
        "F1_score": cls["f1_score"]
    })

summary_df = pd.DataFrame(summary_rows)

FINAL_RESULT_FILE = MODEL_DIR / "final_result_summary.csv"
summary_df.to_csv(FINAL_RESULT_FILE, index=False)

display(summary_df)
print("Saved:", FINAL_RESULT_FILE)

,Split,Precision@5,Recall@5,NDCG@5,Precision@10,Recall@10,NDCG@10,Accuracy,Precision_cls,Recall_cls,F1_score
0,Train,0.190236,0.951180,0.800605,0.099969,0.999690,0.816988,0.931811,0.624961,0.624961,0.624961
1,Valid,0.185093,0.925466,0.755587,0.099938,0.999379,0.780551,0.920497,0.562733,0.562733,0.562733
2,Test,0.187089,0.935444,0.754323,0.099876,0.998759,0.775516,0.917725,0.547486,0.547486,0.547486


Saved: C:\D drive\item_recommendation\data\model_outputs_v2\final_result_summary.csv


In [28]:
# =========================================
# 27. SHORT FINAL PRINT
# =========================================
print("\n========== UPDATED MODEL SUMMARY ==========")
print("\nRanking at 10")
print("Train:", evaluate_grouped_ranking(train_meta, k=10))
print("Valid:", evaluate_grouped_ranking(valid_meta, k=10))
print("Test :", evaluate_grouped_ranking(test_meta, k=10))

print("\nAuxiliary classification")
print("Train:", classification_metrics(train_cls))
print("Valid:", classification_metrics(valid_cls))
print("Test :", classification_metrics(test_cls))


========== UPDATED MODEL SUMMARY ==========

Ranking at 10
Train: {'Precision@10': 0.0999689537410742, 'Recall@10': 0.999689537410742, 'NDCG@10': 0.8169883713415078}
Valid: {'Precision@10': 0.09993788819875776, 'Recall@10': 0.9993788819875776, 'NDCG@10': 0.780550532704836}
Test : {'Precision@10': 0.09987585350713842, 'Recall@10': 0.9987585350713842, 'NDCG@10': 0.7755161089928665}

Auxiliary classification
Train: {'accuracy': 0.9318111258502442, 'precision': 0.6249611921763427, 'recall': 0.6249611921763427, 'f1_score': 0.6249611921763427}
Valid: {'accuracy': 0.9204968944099379, 'precision': 0.5627329192546584, 'recall': 0.5627329192546584, 'f1_score': 0.5627329192546584}
Test : {'accuracy': 0.9177247333671915, 'precision': 0.547486033519553, 'recall': 0.547486033519553, 'f1_score': 0.547486033519553}
